# Sprint 2 — Component 2 TB Head Training

Trains the TorchXRayVision routing head + TB classification head.

**What changed in Sprint 2:**
- Separate LR groups: routing head at `lr`, TB head at `lr × 5.0` (fresh Linear needs faster LR)
- `pos_weight = n_neg / n_pos` in BCE loss to handle TB/normal class imbalance
- Early stopping now monitors val TB AUROC (not combined loss)
- Per-epoch AUROC printed — you can see the head actually learning
- epochs=30, patience=10, tb_head_weight=2.0

**Datasets to attach:**
- `iahmedhabib/montgomery`
- `iahmedhabib/shehzhenn`
- `usmanshams/tbx-11`

NIH is not needed — it has no TB labels and is excluded from training.

In [ ]:
# Cell 1 — Clone repo and install dependencies
import subprocess, sys, os

REPO_DIR = '/kaggle/working/dl-project'

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ['git', 'clone', 'https://github.com/mabdullahi7780/dl-project-codebase.git', REPO_DIR],
        check=True
    )
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')

!pip install -q -r requirements.txt
print('Dependencies installed.')

In [ ]:
# Cell 2 — Verify dataset paths
from pathlib import Path

MONTGOMERY_ROOT = '/kaggle/input/datasets/iahmedhabib/montgomery/montgomery'
SHENZHEN_ROOT   = '/kaggle/input/datasets/iahmedhabib/shehzhenn/shenzhen'
TBX11K_ROOT     = '/kaggle/input/datasets/usmanshams/tbx-11/TBX11K'

def _find(*candidates):
    for c in candidates:
        p = Path(c)
        if p.exists():
            return str(p)
    return None

def _show(name, path, check_subdirs=None):
    p = Path(path) if path else None
    if p and p.exists():
        n = sum(1 for _ in p.rglob('*') if _.is_file())
        print(f'  {name}: OK  ({n} files)  {path}')
        if check_subdirs:
            for sub in check_subdirs:
                sp = p / sub
                print(f'    [{"OK" if sp.exists() else "MISSING"}] {sub}/')
    else:
        print(f'  {name}: NOT FOUND — {path}')
        # Show parent directory to help diagnose
        parent = Path(path).parent if path else None
        if parent and parent.exists():
            print(f'    Parent {parent} contains:')
            for child in sorted(parent.iterdir())[:8]:
                print(f'      {child.name}/')

print('Dataset roots:')
_show('Montgomery', MONTGOMERY_ROOT, ['MontgomerySet/CXR_png', 'CXR_png'])
_show('Shenzhen',   SHENZHEN_ROOT,   ['CXR_png', 'images', 'images/images'])
_show('TBX11K',     TBX11K_ROOT,     ['imgs/tb', 'imgs/health', 'lists'])

# Shenzhen image discovery sanity check
shen = Path(SHENZHEN_ROOT)
for sub in ('CXR_png', 'images/images', 'images'):
    p = shen / sub
    if p.exists():
        imgs = list(p.glob('*.png')) + list(p.glob('*.jpg'))
        print(f'  Shenzhen images found at {sub}/: {len(imgs)}')
        break

In [ ]:
# Cell 3 — Set environment variables and run training
import os

SAVE_DIR = '/kaggle/working/component2_runs'
LOG_FILE = '/kaggle/working/component2_training_log.txt'

os.environ['MONTGOMERY_ROOT']  = MONTGOMERY_ROOT
os.environ['SHENZHEN_ROOT']    = SHENZHEN_ROOT
os.environ['TBX11K_ROOT']      = TBX11K_ROOT
os.environ['COMPONENT2_SAVE_DIR'] = SAVE_DIR

os.makedirs(SAVE_DIR, exist_ok=True)

print('Environment:')
for k in ('MONTGOMERY_ROOT', 'SHENZHEN_ROOT', 'TBX11K_ROOT', 'COMPONENT2_SAVE_DIR'):
    print(f'  {k} = {os.environ[k]}')
print()
print('Starting training...')
print('=' * 60)

In [ ]:
# Cell 4 — Run training (output captured in this cell and also saved to log file)
!python -m src.training.train_component2_txv \
    --config configs/component2_txv.kaggle.yaml \
    --paths configs/paths.yaml \
    2>&1 | tee /kaggle/working/component2_training_log.txt

In [ ]:
# Cell 5 — Verify checkpoint and parse AUROC table
import torch
from pathlib import Path

ckpt_path = Path('/kaggle/working/component2_runs/component2_routing_head.pt')

if ckpt_path.exists():
    print(f'Checkpoint saved: {ckpt_path} ({ckpt_path.stat().st_size / 1e6:.1f} MB)')
    state = torch.load(ckpt_path, map_location='cpu')
    keys = list(state.keys())
    routing_keys = [k for k in keys if k != 'tb_head']
    tb_head_saved = 'tb_head' in state and isinstance(state['tb_head'], dict)
    print(f'  Routing head keys: {len(routing_keys)}')
    print(f'  TB head saved:     {tb_head_saved}')
    if tb_head_saved:
        print(f'  TB head keys:      {list(state["tb_head"].keys())}')
else:
    print('ERROR: checkpoint not found at', ckpt_path)

# Parse AUROC table from log
log_path = Path('/kaggle/working/component2_training_log.txt')
if log_path.exists():
    print()
    print('Training summary (AUROC per epoch):')
    print('-' * 60)
    with open(log_path) as f:
        for line in f:
            line = line.rstrip()
            # Print header, divider, epoch rows, and summary lines
            if any(x in line for x in ['Ep', '---', 'pos:', 'saved', 'AUROC', 'complete', 'Early', 'Best']):
                print(line)

In [ ]:
# Cell 6 — Quick per-dataset val AUROC breakdown
# Loads the saved checkpoint into the model and computes per-dataset AUROC
import sys, os
sys.path.insert(0, '/kaggle/working/dl-project')

import torch
from pathlib import Path
from sklearn.metrics import roc_auc_score
from collections import defaultdict

from src.components.component2_txv import Component2SoftDomainContext
from src.training.train_component2_txv import (
    Component2DomainDataset, collate_component2_batch, stratified_split
)
from src.training.train_component1_dann import (
    build_component1_manifest, load_yaml_config, maybe_limit_manifest
)
from torch.utils.data import DataLoader

CKPT = '/kaggle/working/component2_runs/component2_routing_head.pt'

if not Path(CKPT).exists():
    print('Checkpoint not found — run Cell 4 first.')
else:
    config = load_yaml_config('configs/component2_txv.kaggle.yaml')['component2_txv']
    samples = build_component1_manifest()
    excluded = {d.lower() for d in config['data'].get('exclude_datasets', [])}
    samples = [s for s in samples if s.dataset_id not in excluded]
    _, val_samples = stratified_split(samples, val_ratio=config['data']['val_split'], seed=1337)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = Component2SoftDomainContext(backend='auto', weights='densenet121-res224-all').to(device)
    model.load_trained_routing_head(CKPT)
    model.eval()

    val_loader = DataLoader(
        Component2DomainDataset(val_samples),
        batch_size=32, shuffle=False, num_workers=2,
        collate_fn=collate_component2_batch,
    )

    by_dataset = defaultdict(lambda: {'logits': [], 'labels': []})

    with torch.no_grad():
        for batch in val_loader:
            x = batch['x_224'].to(device)
            labels = batch['tb_label']
            logits = model.forward_tb_logit(x).squeeze(1).cpu()
            for i, ds in enumerate(batch['dataset_id']):
                by_dataset[ds]['logits'].append(logits[i].item())
                by_dataset[ds]['labels'].append(labels[i].item())

    print(f'{"Dataset":<14} {"N_val":>6} {"N_TB":>6} {"AUROC":>8}')
    print('-' * 38)
    for ds, data in sorted(by_dataset.items()):
        lbls = [l for l in data['labels'] if l != -1]
        lgs  = [data['logits'][i] for i, l in enumerate(data['labels']) if l != -1]
        n_tb = sum(1 for l in lbls if l == 1)
        if len(set(lbls)) >= 2:
            import numpy as np
            auc = roc_auc_score(lbls, torch.sigmoid(torch.tensor(lgs)).numpy())
            print(f'{ds:<14} {len(lbls):>6} {n_tb:>6} {auc:>8.4f}')
        else:
            print(f'{ds:<14} {len(lbls):>6} {n_tb:>6} {"N/A (one class)":>15}')